# Milestone 2 — applied refined models
One refinement cycle: conservative reference → incremental feature tests → taught-family model comparison → final application.
Current version: `refinement-selected-v3` (13-input decision tree;14-input regularised logistic). Base27 features are unchanged.
This notebook defaults to reading accepted evidence and no-fit scoring. Earlier notebooks05–07 are historical random-group experiments.
To reproduce, choose a NEW run directory and explicitly enable build. Terminal evaluation is previously inspected diagnostic evidence, never another selection step.


In [ ]:
from pathlib import Path
import json, os, sys, subprocess
import pandas as pd
root = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
assert (root / 'src/models/refinement_selected.py').exists(), 'Run from repo root or notebooks/'
os.chdir(root)
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['OMP_NUM_THREADS'] = '1'
os.environ.setdefault('MPLCONFIGDIR', str(root / 'artifacts/generated/mpl-cache'))
RUN = root / 'artifacts/metrics/refinement-selected-v3'
BUILD_NEW_RUN = False
EVALUATE_PREVIOUSLY_INSPECTED_TERMINAL = False


## Reproduction (optional,19 fixed-choice fits; no search)
Set `RUN` to an unused path, then `BUILD_NEW_RUN=True`. Commands refuse existing outputs. Pin versions from the root README. Verification checks every saved fold prediction and training-only preprocessing. Private course/review/report folders are not required.


In [ ]:
def command(module, *args):
    subprocess.run([sys.executable, '-m', module, *map(str,args)], check=True, cwd=root)
if BUILD_NEW_RUN:
    command('src.models.refinement_selected', 'build', '--output', RUN)
    command('src.models.refinement_selected_verify', '--run', RUN)
    command('src.models.refinement_diagnostics', '--run', RUN)
config = json.loads((RUN / 'configuration.json').read_text())
display(pd.read_csv(RUN / 'development_fold_metrics.csv'))


## Explicit later-period diagnostic (no fitting)
Previously explored data is not a fresh test. Models/features/policy are already frozen. Outcomes must not trigger another change. This command runs only if enabled, and refuses existing terminal output.


In [ ]:
if EVALUATE_PREVIOUSLY_INSPECTED_TERMINAL:
    command('src.models.refinement_selected_terminal', '--run', RUN, '--acknowledge-previously-inspected-terminal')
    command('src.models.refinement_selected_terminal_verify', '--run', RUN)
if (RUN / 'terminal/metrics.csv').exists():
    display(pd.read_csv(RUN / 'terminal/metrics.csv'))


## Strict no-fit scoring on10 preterminal training examples
Only IDs and the selected task inputs enter scoring. Null cells are allowed; missing/extra columns or bad IDs fail. Complete trusted pipelines already include preprocessing. No seller histories or outcome fields are predictors.


In [ ]:
from src.models.refinement_selected import SelectedScorer
base = pd.read_csv(root / 'data/business/ml/orders_ml_features.csv', dtype={'order_id':'string'})
for task, cols in config['predictors_by_task'].items():
    ids = pd.read_csv(RUN / f'{task}_training_ids.csv', dtype={'order_id':'string'}).order_id.iloc[:10]
    batch = base.set_index('order_id').loc[ids, cols].reset_index()
    display(SelectedScorer(RUN / 'bundles' / task / 'selected').score(batch))


## Interpretation and audit trail
Development plots, raw-feature permutation losses, tree top levels and logistic coefficients: `RUN/development_diagnostics`.
All candidate trials—including unsuccessful ones—are losslessly archived under `artifacts/metrics/refinement-cycle1-evidence`; unpack using `python -m experiments.refinement_cycle1.evidence unpack --output NEW_DIRECTORY`. Trial code/protocols are in `experiments/feature_selection_v3` and `experiments/refinement_cycle1`.
The full432-fit route is `python -m experiments.refinement_cycle1.reproduce --output artifacts/generated/full-check-01 --acknowledge-previously-inspected-terminal`. The expansion was adaptive, not an entirely preplanned cycle. Consult the reproduction README for final-versus-trials navigation.
Terminal tree MAE5.5401 is worse than dummy4.8878; do not hide this or select the terminal-winning linear reference. Classifier AP0.19545 is modest. Neither model is deployment-ready.
